<a href="https://colab.research.google.com/github/jcorozcor/Unit-01---Python-Code-Along/blob/main/Module%201.4%20-%20files%20regex%20oop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Juan Module 1.4 — Files, Regular Expressions, and Object-Oriented Programming

## Data Science Bootcamp | Interactive Code-Along

> **Today’s outcome:** You will read and write simple files, process patient-style CSV records, validate text with regular expressions, and model a real-world object using a Python class.

### Scenario for today
A small clinic needs a simple program to manage appointment data. We will use fictional patients and generic health-administration information only. The examples are for programming practice—not clinical advice, diagnosis, or real patient-data handling.

### Learning objectives
By the end of this module, you can:
- Read, write, and append text files using `with open(...)`.
- Explain why context managers safely close files.
- Read and write CSV records using `csv.DictReader` and `csv.DictWriter`.
- Use `rstrip()` to remove trailing newline characters.
- Use regular expressions for simple text validation and cleaning.
- Use `re.search()`, `re.fullmatch()`, and `re.sub()`.
- Define classes, create objects, use `__init__`, and write instance methods.
- Use `@property` validation, `@classmethod`, `__str__`, and simple inheritance.

### Why this matters in data science
Data work begins with files, messy text, and real-world entities. These skills prepare you to load raw records safely, clean them consistently, and design code that represents meaningful objects and rules.

## 1. File input and output

A file lets a program keep information after the program stops running. Python commonly uses `open()` to access a file.

| Mode | Meaning | Typical use |
|---|---|---|
| `'r'` | Read | Read an existing file |
| `'w'` | Write | Create or replace a file |
| `'a'` | Append | Add content to the end of a file |

Use a context manager: `with open(...) as file:`. It closes the file automatically, even if something goes wrong inside the code block.

```python
with open('notes.txt', 'w', encoding='utf-8') as file:
    file.write('Appointment booked.\n')
```

For real systems, never store sensitive personal or medical data in unprotected local files. Use anonymized teaching data and follow organizational security, privacy, and retention policies.

In [ ]:
from pathlib import Path

# This is fictional administrative data for learning only.
notes_path = Path('clinic_notes.txt')

with open(notes_path, 'w', encoding='utf-8') as file:
    file.write('Clinic opening checklist\n')
    file.write('Confirm appointment schedule.\n')
    file.write('Prepare reception desk.\n')

print(f'Created: {notes_path}')

In [ ]:
with open(notes_path, 'r', encoding='utf-8') as file:
    notes = file.readlines()

for note in notes:
    # rstrip() removes the trailing newline added when the line was read.
    print(f'- {note.rstrip()}')

In [ ]:
with open(notes_path, 'a', encoding='utf-8') as file:
    file.write('Check that anonymized training data is used.\n')

with open(notes_path, 'r', encoding='utf-8') as file:
    print(file.read())

### Checkpoint 1
Create a file named `daily_tasks.txt`. Write three simple tasks for a clinic receptionist, append one more task, then read the file and print each task with a number.

**Hint:** Use `enumerate(tasks, start=1)` after reading the lines.

In [ ]:
# Your code here
# from pathlib import Path
# tasks_path = Path('daily_tasks.txt')

## 2. CSV files: rows and columns

CSV means **comma-separated values**. A CSV file is commonly used for simple tabular data. `csv.DictReader` reads each row as a dictionary using the column headers as keys. `csv.DictWriter` writes dictionaries as rows.

This is a useful bridge to pandas: later, pandas will load CSV files into a DataFrame, but the underlying idea is still rows, columns, values, and missing data.

In [ ]:
import csv
from pathlib import Path

appointments_path = Path('appointments.csv')

appointment_rows = [
    {'appointment_id': 'A001', 'patient_name': 'Alex Morgan', 'department': 'General Practice', 'status': ' booked '},
    {'appointment_id': 'A002', 'patient_name': 'Sam Lee', 'department': 'Physiotherapy', 'status': 'COMPLETED'},
    {'appointment_id': 'A003', 'patient_name': 'Jordan Patel', 'department': 'General Practice', 'status': 'cancelled'}
]

fieldnames = ['appointment_id', 'patient_name', 'department', 'status']

with open(appointments_path, 'w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(appointment_rows)

print(f'Created: {appointments_path}')

In [ ]:
with open(appointments_path, 'r', newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row)

## Code-Along 1.4.1 — Clean and write appointment records

### Goal
Read fictional appointment records, standardize the status values, add a simple follow-up field, and write cleaned records to a new CSV file.

### Instructor transcript
Say: *We are not changing the original data. We read raw records, apply transparent transformation rules, and write a separate cleaned output. This creates a simple, auditable data pipeline.*

In [ ]:
cleaned_appointments_path = Path('appointments_cleaned.csv')
cleaned_rows = []

with open(appointments_path, 'r', newline='', encoding='utf-8') as input_file:
    reader = csv.DictReader(input_file)

    for row in reader:
        clean_status = row['status'].strip().title()

        if clean_status == 'Completed':
            follow_up_needed = 'No'
        elif clean_status == 'Cancelled':
            follow_up_needed = 'Yes'
        else:
            follow_up_needed = 'Pending'

        cleaned_rows.append({
            'appointment_id': row['appointment_id'],
            'patient_name': row['patient_name'].strip().title(),
            'department': row['department'].strip(),
            'status': clean_status,
            'follow_up_needed': follow_up_needed
        })

output_fieldnames = ['appointment_id', 'patient_name', 'department', 'status', 'follow_up_needed']

with open(cleaned_appointments_path, 'w', newline='', encoding='utf-8') as output_file:
    writer = csv.DictWriter(output_file, fieldnames=output_fieldnames)
    writer.writeheader()
    writer.writerows(cleaned_rows)

print(f'Wrote {len(cleaned_rows)} cleaned records to {cleaned_appointments_path}.')
for row in cleaned_rows:
    print(row)

### Challenge
Add a fictional record with an empty `department` value. Update the cleaning process so that missing or blank departments become `'Unassigned'`.

**Stretch:** Count the number of records in each cleaned status and print a short summary.

In [ ]:
# Your code here
# Add a record, repeat the cleaning logic, and handle blank department values.

## 3. Regular expressions: matching text patterns

A **regular expression** (regex) describes a text pattern. Regex is helpful when basic string methods are not enough—for example, validating an identifier format or standardizing inconsistent spacing.

Use the `re` module. Start with simple patterns and test examples carefully.

| Pattern | Meaning |
|---|---|
| `^` | Start of text |
| `$` | End of text |
| `\d` | One digit |
| `\w` | One word character: letter, digit, or underscore |
| `\s` | Whitespace |
| `+` | One or more of the preceding pattern |
| `*` | Zero or more of the preceding pattern |
| `?` | Zero or one of the preceding pattern |
| `[ABC]` | One character from the listed set |
| `{3}` | Exactly three repetitions |

Use raw strings such as `r'\d+'` so backslashes are interpreted correctly by the regex engine.

In [ ]:
import re

appointment_id = 'APT-1042'
pattern = r'^APT-\d{4}$'

is_valid = re.fullmatch(pattern, appointment_id) is not None
print(f'{appointment_id}: valid format? {is_valid}')

for candidate in ['APT-1042', 'APT-12', 'apt-1042', '1042']:
    print(f'{candidate!r}: {re.fullmatch(pattern, candidate) is not None}')

### `search()` versus `fullmatch()`

- `re.search(pattern, text)` looks for a match **anywhere** in the text.
- `re.fullmatch(pattern, text)` requires the **entire** text to match.

For validation, `fullmatch()` is often clearer because it rejects extra unexpected characters.

In [ ]:
message = 'Reminder: appointment APT-1042 is scheduled.'

found_id = re.search(r'APT-\d{4}', message)
print('Found identifier:', found_id.group() if found_id else 'None')

messy_name = '  alex     morgan  '
clean_name = re.sub(r'\s+', ' ', messy_name).strip().title()
print('Cleaned name:', clean_name)

## Code-Along 1.4.2 — Validate simple clinic contact fields

### Goal
Create small validation functions for a fictional appointment identifier and a simplified email-like contact value. These patterns are intentionally simplified for teaching; production email validation and identity verification require more careful policies and often external verification.

In [ ]:
import re

def is_valid_appointment_id(value):
    """Return True for IDs in the form APT-1234."""
    return re.fullmatch(r'APT-\d{4}', value.strip()) is not None

def is_simple_email(value):
    """Return True for a simple teaching-only email pattern."""
    pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'
    return re.fullmatch(pattern, value.strip()) is not None

appointment_examples = ['APT-1001', 'apt-1001', 'APT-99', 'APT-10001']
email_examples = ['alex@example.com', 'sam.lee@clinic.org', 'not-an-email', 'name@domain']

for item in appointment_examples:
    print(f'Appointment ID {item!r}: {is_valid_appointment_id(item)}')

for item in email_examples:
    print(f'Email {item!r}: {is_simple_email(item)}')

### Challenge
Write `normalise_phone_number(value)` that removes spaces, hyphens, and parentheses from a fictional contact number. Then confirm that the cleaned result contains only digits and has between 8 and 15 digits.

**Hint:** `re.sub(r'[^0-9]', '', value)` removes everything that is not a digit.

In [ ]:
# Your code here
# def normalise_phone_number(value):
#     ...

## 4. Object-oriented programming: modeling things

Object-oriented programming (OOP) groups related **data** and **behavior**. A class is a blueprint; an object is a specific instance created from that blueprint.

For example, a `Patient` object can store administrative information and provide methods that describe simple administrative actions. It must not be treated as a clinical system.

| OOP term | Meaning |
|---|---|
| Class | Blueprint that defines data and behavior |
| Object / instance | One created object from a class |
| Attribute | Data stored on an object |
| Method | A function defined inside a class |
| `__init__` | Constructor that initializes a new object |
| `self` | Reference to the current object |

In [ ]:
class Patient:
    """A teaching example for fictional, non-clinical appointment administration."""

    def __init__(self, patient_id, name, department):
        self.patient_id = patient_id
        self.name = name
        self.department = department

    def appointment_summary(self):
        return f'{self.name} ({self.patient_id}) is assigned to {self.department}.'

patient_one = Patient('P-1001', 'Alex Morgan', 'General Practice')
patient_two = Patient('P-1002', 'Sam Lee', 'Physiotherapy')

print(patient_one.appointment_summary())
print(patient_two.appointment_summary())

### Encapsulation with properties
A property lets a class control how an attribute is read or changed. This is useful when a value must follow simple rules. In the example below, a patient ID must use the fictional format `P-` followed by four digits.

Use a leading underscore such as `_patient_id` for the internal attribute; external code accesses the validated `patient_id` property.

In [ ]:
class Patient:
    """A teaching-only class for fictional appointment administration."""

    def __init__(self, patient_id, name, department='Unassigned'):
        self.patient_id = patient_id
        self.name = name.strip().title()
        self.department = department.strip().title()

    @property
    def patient_id(self):
        return self._patient_id

    @patient_id.setter
    def patient_id(self, value):
        if not re.fullmatch(r'P-\d{4}', value):
            raise ValueError('Patient ID must follow the fictional format P-1234.')
        self._patient_id = value

    def appointment_summary(self):
        return f'{self.name} ({self.patient_id}) — {self.department}'

    def __str__(self):
        return self.appointment_summary()

patient = Patient('P-1003', '  jordan patel  ', 'general practice')
print(patient)

# Uncomment to observe validation:
# invalid_patient = Patient('1003', 'Jordan Patel')

### Class methods
A class method receives the class itself as `cls`. It is useful for alternate constructors—methods that build an object from a particular kind of input.

In [ ]:
class Patient:
    """A teaching-only class for fictional appointment administration."""

    def __init__(self, patient_id, name, department='Unassigned'):
        self.patient_id = patient_id
        self.name = name.strip().title()
        self.department = department.strip().title()

    @property
    def patient_id(self):
        return self._patient_id

    @patient_id.setter
    def patient_id(self, value):
        if not re.fullmatch(r'P-\d{4}', value):
            raise ValueError('Patient ID must follow the fictional format P-1234.')
        self._patient_id = value

    @classmethod
    def from_csv_row(cls, row):
        """Create a Patient from a dictionary read from a CSV row."""
        return cls(
            patient_id=row['patient_id'],
            name=row['patient_name'],
            department=row.get('department', 'Unassigned')
        )

    def __str__(self):
        return f'{self.name} ({self.patient_id}) — {self.department}'

sample_row = {
    'patient_id': 'P-1004',
    'patient_name': 'mila jovanovic',
    'department': 'physiotherapy'
}

patient_from_row = Patient.from_csv_row(sample_row)
print(patient_from_row)

## Code-Along 1.4.3 — Build an appointment class

### Goal
Create an `Appointment` class that validates a fictional appointment ID, stores a patient name and department, and provides a readable summary.

### Instructor transcript
Say: *The class is not a database and not a clinical system. It is a small model that helps us see how related attributes and actions can be kept together in code.*

In [ ]:
class Appointment:
    """A simple teaching model for fictional appointment administration."""

    allowed_statuses = {'Booked', 'Completed', 'Cancelled'}

    def __init__(self, appointment_id, patient_name, department, status='Booked'):
        self.appointment_id = appointment_id
        self.patient_name = patient_name.strip().title()
        self.department = department.strip().title()
        self.status = status

    @property
    def appointment_id(self):
        return self._appointment_id

    @appointment_id.setter
    def appointment_id(self, value):
        if not re.fullmatch(r'APT-\d{4}', value):
            raise ValueError('Appointment ID must follow the format APT-1234.')
        self._appointment_id = value

    @property
    def status(self):
        return self._status

    @status.setter
    def status(self, value):
        cleaned_status = value.strip().title()
        if cleaned_status not in self.allowed_statuses:
            raise ValueError(f'Status must be one of: {sorted(self.allowed_statuses)}')
        self._status = cleaned_status

    def mark_completed(self):
        self.status = 'Completed'

    def __str__(self):
        return f'{self.appointment_id}: {self.patient_name} | {self.department} | {self.status}'

appointment = Appointment('APT-1001', 'alex morgan', 'general practice')
print(appointment)

appointment.mark_completed()
print(appointment)

### Inheritance: a focused example
Inheritance lets a specialized class reuse features from a more general class. Use it only when the specialized object truly *is a kind of* the general object.

Here, `OnlineAppointment` is an appointment with an additional meeting link. `super()` calls the parent class constructor.

In [ ]:
class OnlineAppointment(Appointment):
    """A fictional appointment with an online meeting link."""

    def __init__(self, appointment_id, patient_name, department, meeting_link, status='Booked'):
        super().__init__(appointment_id, patient_name, department, status)
        self.meeting_link = meeting_link

    def __str__(self):
        return f'{super().__str__()} | Online link: {self.meeting_link}'

online_appointment = OnlineAppointment(
    'APT-1002',
    'sam lee',
    'physiotherapy',
    'https://example.com/meeting/apt-1002'
)

print(online_appointment)

### Challenge
Add a `reschedule(new_department)` method to `Appointment`. The method should update the department and return a short confirmation message.

**Stretch:** Add a class method called `from_dict()` that creates an appointment from a dictionary.

In [ ]:
# Your code here
# Add methods to a new or revised Appointment class.

## Independent practice — Appointment file to objects

Create a small end-to-end workflow using only fictional records:

1. Create a CSV file with at least three appointment rows.
2. Read each row with `csv.DictReader`.
3. Validate each appointment ID with your regex or class property.
4. Convert valid rows into `Appointment` objects.
5. Print each appointment object.
6. Count and report invalid rows rather than silently ignoring them.

**Stretch:** Write valid appointments to a cleaned CSV output file.

In [ ]:
practice_rows = [
    {'appointment_id': 'APT-2001', 'patient_name': 'Taylor Kim', 'department': 'General Practice', 'status': 'Booked'},
    {'appointment_id': 'INVALID', 'patient_name': 'Riley Chen', 'department': 'Physiotherapy', 'status': 'Completed'},
    {'appointment_id': 'APT-2003', 'patient_name': 'Casey Brown', 'department': 'Reception', 'status': 'Cancelled'}
]

# Your code here
# valid_appointments = []
# invalid_rows = []
# ...

## Debugging checklist

### File errors
- Confirm the file name and working directory.
- Use `Path.cwd()` to inspect the current working folder if needed.
- Use `'r'` only when the file already exists.
- Remember that `'w'` replaces an existing file.
- Use `newline=''` with the `csv` module when writing CSV files.

### Regex errors
- Test one valid and several invalid examples.
- Prefer `fullmatch()` for validation.
- Use raw string patterns such as `r'\d+'`.
- Keep patterns simple until requirements demand more complexity.

### OOP errors
- Check that `__init__` is spelled with two underscores on each side.
- Include `self` as the first parameter of instance methods.
- Check indentation inside class definitions.
- Validate values at the boundary where they enter the object.
- Prefer simple classes with a clear responsibility over overly complex designs.

## Check your knowledge!
Answer in your own words:

1. Why is `with open(...)` safer than opening a file without a context manager?
2. What advantage does `csv.DictReader` have over reading CSV rows only as lists?
3. When is `re.fullmatch()` more appropriate than `re.search()`?
4. What is the relationship between a class and an object?
5. Why is it useful to validate an appointment ID when it is assigned to an object?
6. Why should transformations write to a cleaned output file instead of overwriting raw data?

## Preview: Module 1.5
Next, you will use sets, type hints, docstrings, command-line parsing, argument unpacking, comprehensions, and generators to write more expressive and efficient Python.